# 05 — Explainability & Scalability

**Bölüm A:** SHAP ile hangi feature'ların fraud kararını etkilediğini göster (IEEE-CIS)

**Bölüm B:** PaySim (6M+ işlem) üzerinde scalability testi — graph oluşturma ve inference süresi

In [ ]:
import sys, os, time, json
sys.path.insert(0, os.path.abspath('..'))

import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import yaml

if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
print(f'Device: {device}')

---
## Bölüm A: Explainability (SHAP)

Eğitilmiş Hybrid GAT+VAE modelini yükleyip, en yüksek fraud skorlu işlemlere SHAP uyguluyoruz.

In [ ]:
# Veri ve modeli yükle
from src.models.hybrid_model import HybridGATVAE

data = torch.load('../data/processed/ieee_cis/hetero_graph.pt', weights_only=False)

with open('../configs/default.yaml') as f:
    cfg = yaml.safe_load(f)

in_channels = {ntype: data[ntype].x.shape[1] for ntype in data.node_types}

model = HybridGATVAE(
    metadata=data.metadata(),
    in_channels=in_channels,
    gat_hidden=cfg['model']['gat_hidden'],
    gat_out=cfg['model']['gat_out'],
    gat_heads=cfg['model']['gat_heads'],
    gat_layers=cfg['model']['gat_layers'],
    vae_latent=cfg['model']['vae_latent'],
    vae_hidden=cfg['model']['vae_hidden'],
    dropout=cfg['model']['dropout'],
)

model.load_state_dict(torch.load('../results/models/hybrid_gatvae_ieee_cis.pt', weights_only=True))
model.eval()
print('Model yüklendi.')

In [ ]:
# GAT embeddings üret
with torch.no_grad():
    outputs = model(data.x_dict, data.edge_index_dict)

h_t = outputs['h_t'].numpy()  # transaction embeddings (n_txn, 32)
fraud_scores = outputs['fraud_score'].numpy()

print(f'Embedding boyutu: {h_t.shape}')
print(f'En yüksek fraud skor: {fraud_scores.max():.4f}')
print(f'En düşük fraud skor: {fraud_scores.min():.4f}')

In [ ]:
import shap

# SHAP için: en yüksek skorlu 10 işlem + 50 random background
top_fraud_idx = np.argsort(fraud_scores)[-10:]
bg_idx = np.random.RandomState(42).choice(len(h_t), size=50, replace=False)

background = h_t[bg_idx]
explain_samples = h_t[top_fraud_idx]

# Predict fonksiyonu: embedding -> fraud score
def predict_fn(h_t_np):
    h_t_tensor = torch.tensor(h_t_np, dtype=torch.float32)
    with torch.no_grad():
        x_recon, mu, logvar = model.vae(h_t_tensor)
        recon_err = (h_t_tensor - x_recon).pow(2).mean(dim=1, keepdim=True)
        cls_input = torch.cat([h_t_tensor, recon_err], dim=1)
        logit = model.classifier(cls_input).squeeze(-1)
        return torch.sigmoid(logit).numpy()

explainer = shap.KernelExplainer(predict_fn, background)
shap_values = explainer.shap_values(explain_samples, nsamples=200)
print(f'SHAP values shape: {shap_values.shape}')

In [ ]:
# SHAP bar plot: ortalama feature önemleri
feature_names = [f'emb_{i}' for i in range(h_t.shape[1])]

mean_abs_shap = np.abs(shap_values).mean(axis=0)
top_k = 15
top_features = np.argsort(mean_abs_shap)[-top_k:]

plt.figure(figsize=(10, 6))
plt.barh(
    [feature_names[i] for i in top_features],
    mean_abs_shap[top_features],
    color='#E74C3C'
)
plt.xlabel('Mean |SHAP value|')
plt.title('Top 15 Most Important Embedding Dimensions for Fraud Detection')
plt.tight_layout()
os.makedirs('../results/figures', exist_ok=True)
plt.savefig('../results/figures/shap_feature_importance.png', dpi=150)
plt.show()

In [ ]:
# Tek bir fraud işlem için waterfall plot
shap.waterfall_plot(
    shap.Explanation(
        values=shap_values[0],
        base_values=explainer.expected_value,
        feature_names=feature_names,
    ),
    max_display=15,
)

---
## Bölüm B: Scalability (PaySim)

6.3M işlemlik PaySim dataseti üzerinde:
1. Graph oluşturma süresi
2. Inference süresi (gerçek zamanlı alert hedefi: < 1 saniye)

In [ ]:
from src.data.paysim_loader import load_raw, filter_fraud_types

print('PaySim yükleniyor...')
t0 = time.time()
df_paysim = load_raw()
load_time = time.time() - t0

print(f'Yükleme süresi: {load_time:.1f}s')
print(f'Toplam işlem: {len(df_paysim):,}')
print(f'Fraud: {df_paysim["isFraud"].sum():,} ({100*df_paysim["isFraud"].mean():.3f}%)')

# TRANSFER + CASH_OUT'a filtrele
df_filtered = filter_fraud_types(df_paysim)
print(f'\nFiltreli (TRANSFER+CASH_OUT): {len(df_filtered):,}')
print(f'Fraud: {df_filtered["isFraud"].sum():,} ({100*df_filtered["isFraud"].mean():.3f}%)')

In [ ]:
# Graph oluşturma süresi
from src.graph.builder import build_hetero_graph

# Bellek için alt örnek al (tam 6M+ graph M1'de sığmayabilir)
sample_sizes = [10_000, 50_000, 100_000, 500_000]
graph_times = []

for n in sample_sizes:
    df_sample = df_filtered.head(n)
    t0 = time.time()
    g = build_hetero_graph(df_sample, dataset='paysim')
    elapsed = time.time() - t0
    graph_times.append(elapsed)
    n_nodes = sum(g[nt].x.shape[0] for nt in g.node_types)
    n_edges = sum(g[et].edge_index.shape[1] for et in g.edge_types)
    print(f'{n:>10,} txns → {n_nodes:>10,} nodes, {n_edges:>10,} edges | {elapsed:.2f}s')

print()

In [ ]:
# Inference latency testi (100K sample graph üzerinde)
test_graph = build_hetero_graph(df_filtered.head(100_000), dataset='paysim')
test_graph['transaction'].y = torch.zeros(test_graph['transaction'].x.shape[0], dtype=torch.long)

in_ch = {ntype: test_graph[ntype].x.shape[1] for ntype in test_graph.node_types}

test_model = HybridGATVAE(
    metadata=test_graph.metadata(),
    in_channels=in_ch,
    gat_hidden=cfg['model']['gat_hidden'],
    gat_out=cfg['model']['gat_out'],
    gat_heads=cfg['model']['gat_heads'],
    gat_layers=cfg['model']['gat_layers'],
    vae_latent=cfg['model']['vae_latent'],
    vae_hidden=cfg['model']['vae_hidden'],
    dropout=cfg['model']['dropout'],
)
test_model.eval()

# Warmup
with torch.no_grad():
    _ = test_model(test_graph.x_dict, test_graph.edge_index_dict)

# Gerçek ölçüm (5 tekrar ortalaması)
latencies = []
for _ in range(5):
    t0 = time.time()
    with torch.no_grad():
        out = test_model(test_graph.x_dict, test_graph.edge_index_dict)
    latencies.append(time.time() - t0)

avg_latency = np.mean(latencies)
print(f'Inference latency (100K transactions):')
print(f'  Ortalama: {avg_latency:.3f}s')
print(f'  Per-transaction: {1000*avg_latency/100_000:.4f}ms')
print(f'  Hedef (< 1s): {"BAŞARILI" if avg_latency < 1.0 else "AŞILDI"}')

In [ ]:
# Scalability grafikleri
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Graph construction süresi
axes[0].plot(sample_sizes, graph_times, 'o-', color='#3498DB', linewidth=2)
axes[0].set_xlabel('Transaction Sayısı')
axes[0].set_ylabel('Süre (saniye)')
axes[0].set_title('Graph Construction Süresi')
axes[0].grid(True, alpha=0.3)

# Inference latency bar
axes[1].bar(['100K\nTransactions'], [avg_latency], color='#E74C3C', width=0.4)
axes[1].axhline(y=1.0, color='green', linestyle='--', label='1s hedefi')
axes[1].set_ylabel('Süre (saniye)')
axes[1].set_title('Inference Latency')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../results/figures/scalability.png', dpi=150)
plt.show()

In [ ]:
# Sonuçları JSON'a kaydet
scalability_results = {
    'graph_construction': {
        'sample_sizes': sample_sizes,
        'times_seconds': graph_times,
    },
    'inference_latency': {
        'num_transactions': 100_000,
        'avg_seconds': avg_latency,
        'per_transaction_ms': 1000 * avg_latency / 100_000,
        'under_1s': avg_latency < 1.0,
    },
}

with open('../results/scalability_results.json', 'w') as f:
    json.dump(scalability_results, f, indent=2)

print('Sonuçlar kaydedildi: results/scalability_results.json')